<div style="display:flex; align-items:center; gap:18px; text-align:left">
  <img src="https://sebastiancontz.github.io/ust-diplomado-ia-curso-intro-ia/assets/logo-ust.svg" width="100" alt="Logo Universidad Santo Tomás">
  <div>
    <p>Diplomado en Inteligencia Artificial para los Negocios</p>
    <p>Facultad de Ingeniería y Negocios</p>
    <p>Módulo 3 · Introducción a la Inteligencia Artificial Generativa</p>
    <p>Semana 04: Consumo de modelos vía API y estructuración de salidas</p>
  </div>
</div>

# Consumo de modelos vía API y estructuración de salidas

Bienvenidas y bienvenidos al taller práctico de la **Clase 04**.

En esta sesión daremos el salto fundamental desde la interfaz gráfica manual (Google AI Studio) hacia la **automatización de procesos** mediante llamadas programáticas con la librería oficial `google-genai`.

### Objetivos del taller
1. **Custodiar credenciales:** Configurar su `GEMINI_API_KEY` mediante el gestor seguro de secretos de Colab sin exponerla en código público.
2. **Conectar la ventanilla oficial:** Inicializar el cliente `google.genai.Client` de la API de Gemini.
3. **Garantizar contratos de datos:** Diseñar un esquema formal con `pydantic` (`BaseModel`, `Field`, `ConfigDict`) y `google.genai.types.GenerateContentConfig` para forzar respuestas en JSON limpio sin errores sintácticos.
4. **Extraer y auditar reclamos:** Procesar un texto desordenado de clientes y obtener un objeto estructurado y tipado listo para integrar en sistemas corporativos (ERP/CRM).

## 1. Preparación del entorno

Instalamos la librería oficial de Google (`google-genai`) y la librería estándar de esquemas de datos (`pydantic`).

In [1]:
# Instalación silenciosa de las librerías necesarias
!pip install -q google-genai pydantic

## 2. Autenticación y custodia de la API Key

Como aprendimos en la sesión teórica, la **API Key es la tarjeta de crédito corporativa** de su departamento. Escribirla en texto plano dentro de un script expone a la empresa a fraudes y facturación imprevista.

### Cómo configurar su clave en Google Colab:
1. En el panel lateral izquierdo de Google Colab, hagan clic en el icono de la llave (**Secrets**).
2. Creen un nuevo secreto con el nombre exacto: `GEMINI_API_KEY`.
3. Peguen el valor de su llave obtenida en [Google AI Studio](https://aistudio.google.com/app/apikey).
4. Activen el interruptor de acceso (*Notebook access*) para este cuaderno.

> **Regla de oro de ciberseguridad:** Jamás peguen su API Key o credenciales en chats web para depurar código. Los modelos de lenguaje no son bóvedas de secretos; las credenciales podrían quedar expuestas en historiales compartidos, extensiones de navegador no auditadas o ataques de inyección.

El siguiente bloque lee la credencial directamente desde la memoria cifrada del entorno, sin exponerla en pantalla:

In [2]:
import os

try:
    from google.colab import userdata
    api_key = userdata.get('GEMINI_API_KEY')
except Exception:
    # Fallback seguro para ejecución local o en servidores
    api_key = os.environ.get('GEMINI_API_KEY')

if not api_key:
    print('AVISO: No se detectó GEMINI_API_KEY en los secretos. Configure la llave en el panel lateral de Colab.')
else:
    print('Credencial verificada con éxito desde el gestor seguro de secretos.')

AVISO: No se detectó GEMINI_API_KEY en los secretos. Configure la llave en el panel lateral de Colab.


## 3. Inicialización del cliente oficial (`google.genai.Client`)

La clase `Client` del paquete `google.genai` actúa como nuestra **ventanilla única de atención**. Es el canal estandarizado que se encarga de empaquetar nuestras solicitudes, adjuntar las credenciales de autenticación y comunicarse con los centros de datos de Google mediante protocolos seguros HTTPS.

In [3]:
from google import genai
from google.genai import types

# Inicializar el canal de comunicación oficial
if api_key:
    client = genai.Client(api_key=api_key)
    print('Cliente oficial google-genai inicializado y autenticado.')
else:
    client = None
    print('AVISO: Cliente en modo demostración local (sin conexión activa a la API).')

AVISO: Cliente en modo demostración local (sin conexión activa a la API).


## 4. Definición del contrato de datos con Pydantic

Para evitar que el modelo responda con textos conversacionales, comas faltantes o claves arbitrarias, definimos un **formulario notarial preimpreso** utilizando `pydantic.BaseModel`.

Exigiremos exactamente los 4 casilleros de la ficha funcional de control interno:
- `categoria`: Dominio cerrado restringido a `'facturación'`, `'servicio'` o `'producto'` (`Literal`).
- `monto`: Cifra monetaria en pesos chilenos con cota no negativa $\ge 0.0$ (`float`, `ge=0.0`).
- `urgencia`: Nivel de prioridad operativa restringido a `'baja'`, `'media'` o `'alta'` (`Enum`).
- `escalar_jefatura`: Booleano estricto (`bool`) que indica si el caso debe alertar a una jefatura.
- `model_config = ConfigDict(extra='forbid', strict=True)`: Bloquea cualquier campo adicional e impone tipado estricto sin coerción automática.

### Enfoque low-code: construyan su esquema con asistencia de IA

Como profesionales de control de gestión, auditoría y finanzas, su rol principal consiste en especificar con precisión las reglas de negocio y los tipos requeridos, no en memorizar la sintaxis de Python.

**Ayúdense con el siguiente prompt para generar este esquema en Google AI Studio o Gemini:**

> Actúa como desarrollador Python especializado en extracción estructurada para auditoría y control de gestión.
> Genera una clase Pydantic v2 para estructurar y validar la extracción de reclamos de clientes según este requerimiento de negocio:
>
> 1. Nombre de la clase principal: AuditoriaReclamo (hereda de BaseModel).
> 2. Configuración estricta: model_config = ConfigDict(extra='forbid', strict=True) para rechazar casilleros no autorizados y forzar tipado estricto sin coerción automática.
> 3. categoria: dominio cerrado con typing.Literal['facturación', 'servicio', 'producto'] y Field(description='Clasificación estricta').
> 4. monto: número decimal (float) con cota no negativa (ge=0.0) para el monto en disputa en pesos chilenos (CLP), o 0.0 si no aplica.
> 5. urgencia: catálogo cerrado mediante una clase Urgencia(str, Enum) con opciones 'baja', 'media' o 'alta'.
> 6. escalar_jefatura: booleano (bool) que indique si amerita revisión de jefatura.
>
> Entrega únicamente el bloque de código en Python con los imports necesarios (BaseModel, Field, ConfigDict, Enum, Literal), limpio y listo para ejecutar.

El código generado por el asistente corresponde exactamente a la celda siguiente:

In [4]:
from enum import Enum
from typing import Literal
from pydantic import BaseModel, Field, ConfigDict

class Urgencia(str, Enum):
    BAJA = 'baja'
    MEDIA = 'media'
    ALTA = 'alta'

class AuditoriaReclamo(BaseModel):
    model_config = ConfigDict(extra='forbid', strict=True)

    categoria: Literal['facturación', 'servicio', 'producto'] = Field(
        description='Clasificación estricta: facturación, servicio o producto'
    )
    monto: float = Field(
        ge=0.0,
        description='Monto en disputa en CLP, o 0.0 si no aplica'
    )
    urgencia: Urgencia = Field(
        description='Prioridad operativa asignada'
    )
    escalar_jefatura: bool = Field(
        description='Requiere revisión de jefatura'
    )

## 5. Extracción estructurada: el bloque nuclear

Configuramos la llamada mediante `google.genai.types.GenerateContentConfig` (la clase de configuración avanzada del SDK).

Al definir:
- `response_mime_type='application/json'`
- `response_schema=AuditoriaReclamo`

El decodificador del modelo fuerza matemáticamente la generación para que coincida con el formulario preimpreso. La probabilidad de recibir texto conversacional o campos fuera de norma es **cero**.

In [5]:
# 1. Texto no estructurado recibido del cliente
texto_reclamo = (
    'Estimados, escribo indignada porque en mi factura de este mes me cobraron $45.000 '
    'de mantenimiento técnico que nunca solicité. Si no anulan ese cobro antes del viernes, '
    'cancelaré mi cuenta corporativa y presentaré una denuncia ante el regulador.'
)

# 2. Configuración estricta del contrato de datos
config = types.GenerateContentConfig(
    response_mime_type='application/json',
    response_schema=AuditoriaReclamo,
    temperature=0.0,
)

# 3. Invocación a la API o demostración guiada del esquema
if client:
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=f'Audita el siguiente reclamo y extrae la ficha requerida:\n\n{texto_reclamo}',
        config=config,
    )
    resultado = AuditoriaReclamo.model_validate_json(response.text)
    print('[Ejecución en vivo vía API] Datos extraídos y validados exitosamente:')
else:
    print('[Modo demostración guiado: llave no configurada; resultado esperado por el contrato de datos]')
    resultado = AuditoriaReclamo(
        categoria='facturación',
        monto=45000.0,
        urgencia=Urgencia.ALTA,
        escalar_jefatura=True,
    )

print(resultado)

[Modo demostración guiado: llave no configurada; resultado esperado por el contrato de datos]
categoria='facturación' monto=45000.0 urgencia=<Urgencia.ALTA: 'alta'> escalar_jefatura=True


## 6. Inspección de resultados para control de gestión

Observen cómo el objeto `resultado` contiene atributos de Python directos (`resultado.categoria`, `resultado.monto`). No tenemos que 'cortar' cadenas ni buscar palabras con expresiones regulares; los datos ya están tipados y listos para operar:

In [6]:
print(f'Categoría detectada: {resultado.categoria}')
print(f'Monto reclamado:     ${resultado.monto:,.0f} CLP')
print(f'Prioridad operativa: {resultado.urgencia.value.upper()}')
print(f'Escalar a jefatura:  {"SÍ" if resultado.escalar_jefatura else "NO"}')

Categoría detectada: facturación
Monto reclamado:     $45,000 CLP
Prioridad operativa: ALTA
Escalar a jefatura:  SÍ


## 7. Práctica autónoma: caso borde sin monto explícito

Ahora pongan a prueba la robustez del contrato frente a un caso borde: ¿Qué ocurre si un cliente presenta un reclamo severo pero sin mencionar una cifra de dinero?

Evalúen el siguiente texto:
*"Llevo dos semanas esperando la entrega de los insumos de oficina y nadie contesta los correos. Esto está retrasando la operación completa del equipo."*

### Actividad de control interno (Lo hacen ustedes)
Antes de ejecutar la celda siguiente, completen mentalmente o anoten su predicción en esta pauta de auditoría:

| Campo | Predicción esperada | Fundamento de control interno |
|:---|:---|:---|
| `categoria` | ¿`'servicio'` o `'facturación'`? | ¿Reclama demora en entrega o cobro indebido? |
| `monto` | ¿`0.0` o cifra inventada? | No hay monto explícito. El esquema impone cota $\ge 0.0$, pero que el modelo devuelva `0.0` es una hipótesis sujeta a supervisión humana (el validador no impide que un modelo alucine un monto positivo). |
| `urgencia` | ¿`BAJA`, `MEDIA` o `ALTA`? | Dos semanas sin respuesta operativa en oficina. |
| `escalar_jefatura` | ¿`True` o `False`? | Evaluar impacto en continuidad operacional. |

Ahora ejecuten la celda para contrastar su predicción con la extracción del modelo:

In [7]:
reclamo_operacional = (
    'Llevo dos semanas esperando la entrega de los insumos de oficina y nadie contesta los correos. '
    'Esto está retrasando la operación completa del equipo.'
)

if client:
    response_caso2 = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=f'Audita este reclamo y extrae la ficha requerida:\n\n{reclamo_operacional}',
        config=config,
    )
    resultado_caso2 = AuditoriaReclamo.model_validate_json(response_caso2.text)
    print('[Ejecución en vivo vía API] Resultado del caso borde:')
else:
    print('[Modo demostración guiado: llave no configurada; resultado esperado por el contrato de datos]')
    resultado_caso2 = AuditoriaReclamo(
        categoria='servicio',
        monto=0.0,
        urgencia=Urgencia.MEDIA,
        escalar_jefatura=False,
    )

print(f'Categoría:          {resultado_caso2.categoria}')
print(f'Monto registrado:   ${resultado_caso2.monto:.1f} CLP (0.0 = hipótesis sujeta a supervisión humana)')
print(f'Urgencia asignada:  {resultado_caso2.urgencia.value.upper()}')
print(f'Escalar a jefatura: {"SÍ" if resultado_caso2.escalar_jefatura else "NO"}')
print('Nota de auditoría:  El contrato técnico valida estructura y tipos; la exactitud del contenido requiere supervisión humana.')

[Modo demostración guiado: llave no configurada; resultado esperado por el contrato de datos]
Categoría:          servicio
Monto registrado:   $0.0 CLP (0.0 = hipótesis sujeta a supervisión humana)
Urgencia asignada:  MEDIA
Escalar a jefatura: NO
Nota de auditoría:  El contrato técnico valida estructura y tipos; la exactitud del contenido requiere supervisión humana.


### Transferencia a sus propios procesos de negocio

Para implementar este patrón en sus respectivas áreas (por ejemplo, validación de facturas de proveedores, rendición de viáticos o contratos comerciales), no necesitan programar desde cero.

**Ayúdense con el siguiente prompt para construir esquemas adaptados a su empresa en Google AI Studio o Gemini:**

> Actúa como desarrollador Python especializado en extracción estructurada para control interno.
> Necesito un esquema Pydantic para el proceso: [NOMBRE DEL PROCESO, ej. Rendición de Gastos de Viaje].
>
> Los campos requeridos son:
> - [Campo 1, ej. proveedor]: [Tipo de dato y descripción del negocio]
> - [Campo 2, ej. monto_total]: [Tipo numérico y moneda con cota no negativa]
> - [Campo 3, ej. estado_aprobacion]: [Opciones permitidas en catálogo cerrado Enum]
> - [Campo 4, ej. cumple_politica]: [Booleano True/False]
>
> Genera el código con BaseModel, Field(description="...") y ConfigDict(extra='forbid') listo para configurar con GenerateContentConfig en Gemini API.

## 8. Síntesis y buenas prácticas directivas

Para concluir el taller, retengan estas tres directrices para implementar IA en sus organizaciones:

1. **Seguridad y gobierno:** Nunca expongan credenciales en código compartido. Utilicen gestores de secretos corporativos.
2. **Contratos estrictos:** Utilicen siempre esquemas formales (Pydantic / Schemas) con dominios cerrados y `extra='forbid'` para garantizar que los datos fluyan hacia el ERP sin errores de sintaxis ni casilleros espurios.
3. **Economía de tokens:** Diseñen esquemas concisos para minimizar el costo de los tokens de salida, que son hasta 4 veces más caros que los de entrada.

## Atribución de datos

- **Creador:** Sebastián Contreras
- **Procedencia:** Casos sintéticos elaborados con fines docentes mediante el [script generador](_build_04_apis.py). No representan a clientes, facturas ni organizaciones reales.
- **Modificación:** Ninguna.